# Feature Extraction dengan SIFT, ORB, dan AKAZE

Notebook ini menjelaskan tahap feature extraction pada final project Computer Vision. Tujuan tahap ini adalah mengubah citra menjadi representasi numerik yang dapat dianalisis, dibandingkan, atau digunakan sebagai input untuk Machine Learning.

Pada project ini, feature extraction dilakukan menggunakan tiga metode lokal: SIFT, ORB, dan AKAZE. Setiap metode mendeteksi keypoint dan menghasilkan descriptor. Statistik dari descriptor, seperti jumlah keypoint, rata-rata descriptor, dan variansi descriptor, kemudian disimpan ke `outputs/feature_statistics.csv`.

## Import Library dan Definisi Path

Bagian ini memuat library yang dibutuhkan. OpenCV digunakan untuk membaca citra dan menjalankan detector fitur, pandas digunakan untuk membaca CSV statistik, dan matplotlib digunakan untuk visualisasi.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SAMPLE_DIR = PROJECT_ROOT / "dataset" / "original" / "train" / "real"
FEATURE_CSV_PATH = PROJECT_ROOT / "outputs" / "feature_statistics.csv"

plt.rcParams["figure.figsize"] = (12, 5)

## Konsep Keypoint dan Descriptor

Keypoint adalah titik penting pada citra, misalnya sudut, tepi, atau tekstur lokal yang cukup khas. Descriptor adalah vektor numerik yang menggambarkan pola di sekitar keypoint tersebut.

SIFT biasanya menghasilkan descriptor yang kaya dan stabil, tetapi lebih berat secara komputasi. ORB lebih cepat dan ringan karena menggunakan descriptor biner. AKAZE berada di antara keduanya sebagai alternatif yang juga menggunakan descriptor biner dan nonlinear scale space.

## Load Sample Original Image

Untuk demo, kita menggunakan satu sample citra dari folder `dataset/original/train/real`. Citra ini akan dipakai untuk membandingkan hasil keypoint dari SIFT, ORB, dan AKAZE.

In [ ]:
def find_first_jpg(folder: Path) -> Path:
    """Mengambil file JPG pertama dari sebuah folder."""
    image_paths = sorted(
        {
            path
            for pattern in ("*.jpg", "*.jpeg", "*.JPG", "*.JPEG")
            for path in folder.glob(pattern)
        }
    )
    if not image_paths:
        raise FileNotFoundError(f"Tidak ada citra JPG di {folder}")
    return image_paths[0]


def bgr_to_rgb(image):
    """Mengubah citra BGR OpenCV menjadi RGB untuk matplotlib."""
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)


sample_path = find_first_jpg(SAMPLE_DIR)
sample_bgr = cv2.imread(str(sample_path))
if sample_bgr is None:
    raise ValueError(f"Gagal memuat citra: {sample_path}")

print(f"Sample image: {sample_path}")
plt.imshow(bgr_to_rgb(sample_bgr))
plt.title("Sample original image")
plt.axis("off")
plt.show()

## Konversi ke Grayscale

Detector fitur bekerja pada pola intensitas, sehingga citra dikonversi ke grayscale sebelum keypoint dan descriptor dihitung.

In [ ]:
sample_gray = cv2.cvtColor(sample_bgr, cv2.COLOR_BGR2GRAY)

plt.imshow(sample_gray, cmap="gray")
plt.title("Sample grayscale")
plt.axis("off")
plt.show()

## Menjalankan SIFT, ORB, dan AKAZE

Pada bagian ini, tiga detector dibuat secara eksplisit: `cv2.SIFT_create()`, `cv2.ORB_create(nfeatures=1000)`, dan `cv2.AKAZE_create()`. Setiap detector menjalankan `detectAndCompute()` untuk menghasilkan keypoint dan descriptor.

In [ ]:
detectors = {
    "SIFT": cv2.SIFT_create(),
    "ORB": cv2.ORB_create(nfeatures=1000),
    "AKAZE": cv2.AKAZE_create(),
}

features = {}
for method, detector in detectors.items():
    keypoints, descriptors = detector.detectAndCompute(sample_gray, None)
    features[method] = {
        "keypoints": keypoints,
        "descriptors": descriptors,
    }

feature_summary = pd.DataFrame(
    [
        {
            "method": method,
            "num_keypoints": len(data["keypoints"]),
            "descriptor_shape": None if data["descriptors"] is None else data["descriptors"].shape,
        }
        for method, data in features.items()
    ]
)

display(feature_summary)

## Visualisasi Keypoint

Visualisasi berikut menunjukkan lokasi keypoint pada citra. Perbedaan jumlah dan sebaran keypoint menunjukkan bahwa setiap metode memiliki karakteristik deteksi fitur yang berbeda.

In [ ]:
def draw_keypoints(image_bgr, keypoints):
    """Menggambar keypoint dengan ukuran dan arah orientasi."""
    return cv2.drawKeypoints(
        image_bgr,
        keypoints,
        None,
        flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS,
    )


fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (method, data) in zip(axes, features.items()):
    visualized = draw_keypoints(sample_bgr, data["keypoints"])
    ax.imshow(bgr_to_rgb(visualized))
    ax.set_title(f"{method}: {len(data['keypoints'])} keypoints")
    ax.axis("off")

plt.tight_layout()
plt.show()

## Load Feature Statistics CSV

File `outputs/feature_statistics.csv` berisi hasil ekstraksi fitur dari seluruh kombinasi method, augmentasi, dan label yang diproses oleh `src/feature_extraction.py`.

In [ ]:
feature_df = pd.read_csv(FEATURE_CSV_PATH)

print(f"Rows: {feature_df.shape[0]}")
print(f"Columns: {feature_df.shape[1]}")
display(feature_df.head())

## Tabel Perbandingan Statistik Fitur

Tabel berikut merangkum rata-rata jumlah keypoint, rata-rata descriptor mean, dan rata-rata descriptor variance untuk setiap kombinasi method, augmentasi, dan label.

In [ ]:
comparison_table = (
    feature_df.groupby(["method", "augmentation", "label"], as_index=False)
    .agg(
        average_num_keypoints=("num_keypoints", "mean"),
        average_descriptor_mean=("descriptor_mean", "mean"),
        average_descriptor_variance=("descriptor_variance", "mean"),
    )
    .sort_values(["method", "augmentation", "label"])
)

display(comparison_table)

## Bar Chart Rata-rata Keypoint

Bar chart berikut membandingkan rata-rata jumlah keypoint untuk SIFT, ORB, dan AKAZE. Nilai ini membantu melihat metode mana yang cenderung mendeteksi lebih banyak struktur lokal pada dataset demo.

In [ ]:
average_keypoints = (
    feature_df.groupby("method", as_index=False)["num_keypoints"]
    .mean()
    .sort_values("method")
)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(
    average_keypoints["method"],
    average_keypoints["num_keypoints"],
    color=["#54A24B", "#F58518", "#4C78A8"],
)

ax.set_title("Rata-rata Jumlah Keypoint per Metode")
ax.set_xlabel("Metode")
ax.set_ylabel("Average num_keypoints")

for bar in bars:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{height:.1f}",
        ha="center",
        va="bottom",
    )

plt.tight_layout()
plt.show()

## Kesimpulan

SIFT umumnya lebih deskriptif dan stabil karena descriptor-nya menyimpan informasi local gradient yang kaya, tetapi metode ini lebih berat secara komputasi. ORB lebih cepat dan ringan, sehingga cocok sebagai baseline efisien. AKAZE dapat dipahami sebagai alternatif tengah karena tetap menggunakan descriptor biner, tetapi dengan pendekatan deteksi fitur yang berbeda.

Augmentasi seperti Gaussian blur, Gaussian noise, dan JPEG compression dapat mengubah jumlah serta kualitas keypoint yang terdeteksi. Karena itu, statistik keypoint dan descriptor menjadi informasi penting untuk memahami robustness metode feature extraction sebelum masuk ke tahap matching atau Machine Learning.